In [2]:
import pandas as pd

CSV_PATH = "Ratings.csv"

ratings = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="latin-1")
ratings.columns = [c.strip().lower().replace("-", "_").replace(" ", "_") for c in ratings.columns]

user_col = "user_id"
item_col = "isbn"
rate_col = "rating"

ratings = ratings[[user_col, item_col, rate_col]].dropna()
ratings[rate_col] = pd.to_numeric(ratings[rate_col], errors="coerce")
ratings = ratings.dropna()
ratings = ratings[ratings[rate_col] != 0]

books = pd.Index(sorted(ratings[item_col].unique()), name="isbn")
b2feat = {b: i + 1 for i, b in enumerate(books)}

def to_libsvm_line(df):
    feats = [(b2feat[b], float(r)) for b, r in zip(df[item_col], df[rate_col])]
    feats.sort(key=lambda x: x[0])
    return "0 " + " ".join(f"{j}:{v:g}" for j, v in feats)

lines = (
    ratings.sort_values([user_col, item_col])
           .groupby(user_col, sort=True)
           .apply(to_libsvm_line)
           .tolist()
)

with open("user_book_matrix.libsvm", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Wrote user_book_matrix.libsvm")
print("Users:", ratings[user_col].nunique(), "Books:", len(books))
print("Note: leading 0 is a placeholder label; not used in analysis.")


Wrote user_book_matrix.libsvm
Users: 77805 Books: 185973
Note: leading 0 is a placeholder label; not used in analysis.


/var/folders/g6/7ydynlp56bz41byb4t_npq440000gn/T/ipykernel_57464/216500253.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(to_libsvm_line)
